In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

In [2]:
csv_df = './results/spkr_similarity.csv'
csv_df = pd.read_csv(csv_df)

# drop columns that doesn't contain Answer
csv_df = csv_df.drop(columns=[col for col in csv_df.columns if 'Answer_overall' not in col])
print(csv_df.shape)

(20, 60)


In [7]:
def parse_result(row):
    result = {}
    AB_mapping = {
        'TAO': 0,
        'OAT': 1,
    }

    AX_mapping = {
        'TATd': 1,
        'TdAT': 0,
    }

    BX_mapping = {
        'OATd': 1,
        'TdAO': 0,
    }
    
    for col in csv_df.columns:
        data_name = col.split('Answer.SS_')[1]
        type = data_name.split('_')[0] # TAO: A is baseline and B is our model; OAT: A is our model and B is baseline
        utternace = data_name.replace(type + '_', '').replace('_Answer_overall', '')
        if type in AB_mapping:
            if type == 'TAO':
                result[('AB', utternace)] = AB_mapping['TAO'] if row[col] == 1 else AB_mapping['OAT']
            elif type == 'OAT':
                result[('AB', utternace)] = AB_mapping['OAT'] if row[col] == 1 else AB_mapping['TAO']
        elif type in AX_mapping:
            if type == 'TATd':
                result[('AX', utternace)] = AX_mapping['TATd'] if row[col] == 1 else AX_mapping['TdAT']
            elif type == 'TdAT':
                result[('AX', utternace)] = AX_mapping['TdAT'] if row[col] == 1 else AX_mapping['TATd']
        elif type in BX_mapping:
            if type == 'OATd':
                result[('BX', utternace)] = BX_mapping['OATd'] if row[col] == 1 else BX_mapping['TdAO']
            elif type == 'TdAO':
                result[('BX', utternace)] = BX_mapping['TdAO'] if row[col] == 1 else BX_mapping['OATd']

    return result

In [9]:
result = {
    'AB': [],
    'AX': [],
    'BX': [],
}
for iter, row in csv_df.iterrows():
    _result = parse_result(row)

    for key, value in _result.items():
        result[key[0]].append(value)

In [11]:
key_mapping = {
    'AB': 'TVTSyn vs Ours',
    'AX': 'Target vs TVTSyn',
    'BX': 'Target vs Ours',
}
for key in result:
    result[key] = np.array(result[key])
    print(f'{key_mapping[key]}: {result[key].mean():.2f}')

TVTSyn vs Ours: 0.54
Target vs TVTSyn: 0.38
Target vs Ours: 0.41
